In [2]:
import os
import re
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

# =========================================================================
# Configuration
# =========================================================================

PATH        = r'C:\Users\lidon\Desktop\2026spring\IDX_MLS_Analytics'
SOLD_CSV    = os.path.join(PATH, 'all_sold_residential.csv')
LISTING_CSV = os.path.join(PATH, 'all_listings_residential.csv')
DTYPE_SPEC  = {'PostalCode': str, 'ListingKey': str}

# ID/code columns that must NOT be auto-converted to numeric
# even if all values look like numbers
NUMERIC_CONVERSION_EXCLUDE = {
    'PostalCode', 'ListingId', 'ListingKey', 'ListingKeyNumeric',
    'BuyerAgentMlsId', 'ListAgentEmail',
}

CA_LAT_RANGE = (32, 42)
CA_LON_RANGE = (-125, -114)

# =========================================================================
# Load datasets
# =========================================================================

print("Loading datasets...")
sold_raw     = pd.read_csv(SOLD_CSV,    dtype=DTYPE_SPEC, low_memory=False)
listings_raw = pd.read_csv(LISTING_CSV, dtype=DTYPE_SPEC, low_memory=False)

sold     = sold_raw.copy()
listings = listings_raw.copy()

print(f"  all_sold     : {len(sold):,} rows  |  {sold.shape[1]} columns")
print(f"  all_listings : {len(listings):,} rows  |  {listings.shape[1]} columns\n")

# =========================================================================
# STEP 1 — Convert date columns to datetime
# Learned from teammate: print before/after missing counts so we can see
# how many values failed to parse and became NaT.
# =========================================================================

print("=" * 65)
print("STEP 1 — DATE FORMAT CONVERSION")
print("=" * 65)

DATE_COLS = [
    'CloseDate', 'PurchaseContractDate',
    'ListingContractDate', 'ContractStatusChangeDate',
]

for df, label in [(sold, 'sold'), (listings, 'listings')]:
    print(f"\n  [{label}]")
    for col in DATE_COLS:
        if col in df.columns:
            before_missing = df[col].isna().sum()
            df[col] = pd.to_datetime(df[col], errors='coerce')
            after_missing      = df[col].isna().sum()
            newly_unparseable  = after_missing - before_missing
            print(f"    {col:<35} → {df[col].dtype}  "
                  f"missing before={before_missing:,}, after={after_missing:,} "
                  f"({newly_unparseable} failed to parse → NaT)")
        else:
            print(f"    {col:<35} not found — skipped")

# =========================================================================
# STEP 2 — Convert boolean columns
# All 7 YN columns converted using .astype("boolean").
# WaterfrontYN and BasementYN are NOT dropped here — they will appear
# in Step 4 missing value analysis (~99% and ~98% missing) and be
# dropped there together with other high-missing columns.
# =========================================================================

print("\n" + "=" * 65)
print("STEP 2 — BOOLEAN FORMAT CONVERSION (YN COLUMNS)")
print("=" * 65)

boolean_columns = [
    "AttachedGarageYN", "ViewYN", "PoolPrivateYN",
    "NewConstructionYN", "FireplaceYN",
    "WaterfrontYN", "BasementYN",
]

for col in boolean_columns:
    if col in sold.columns:
        sold[col] = sold[col].astype("boolean")

for col in boolean_columns:
    if col in listings.columns:
        listings[col] = listings[col].astype("boolean")

print("Sold boolean column dtypes:")
print(sold[[col for col in boolean_columns if col in sold.columns]].dtypes)
print("\nList boolean column dtypes:")
print(listings[[col for col in boolean_columns if col in listings.columns]].dtypes)

# =========================================================================
# STEP 3 — Remove duplicate suffix columns (.1/.2/.3)
# Learned from teammate: if the base column is already datetime (converted
# in Step 1), parse the .1 column as datetime too before comparing.
# This fixes CloseDate.1 being flagged as "different" just because
# NaT != NaN even though the underlying data is identical.
# =========================================================================

print("\n" + "=" * 65)
print("STEP 3 — REMOVE DUPLICATE SUFFIX COLUMNS")
print("=" * 65)

def drop_duplicate_suffix_cols(df, label):
    dup_cols = [c for c in df.columns if re.match(r'.+\.\d+$', c)]
    if not dup_cols:
        print(f"\n  [{label}] No duplicate-suffix columns found — clean.")
        return df

    print(f"\n  [{label}] Found {len(dup_cols)} duplicate-suffix columns:")
    cols_to_drop = []

    for c in dup_cols:
        base_col = re.sub(r'\.\d+$', '', c)
        if base_col not in df.columns:
            continue

        a = df[base_col]
        b = df[c]

        # If base column was converted to datetime in Step 1,
        # parse the .1 duplicate the same way before comparing.
        # Otherwise NaT (datetime null) vs NaN (object null) would
        # look like a mismatch even though the data is the same.
        if pd.api.types.is_datetime64_any_dtype(a):
            a_cmp = a
            b_cmp = pd.to_datetime(b, errors='coerce')
        else:
            a_cmp = a.astype(str)
            b_cmp = b.astype(str)

        overlap    = a.notna() & b.notna()
        mismatches = int((a_cmp[overlap] != b_cmp[overlap]).sum())

        if mismatches == 0:
            cols_to_drop.append(c)
            print(f"    • {c}  identical to  {base_col}  → removed")
        else:
            print(f"    ⚠️  {c}  differs from  {base_col}  "
                  f"({mismatches:,} mismatching rows) → kept")

    if cols_to_drop:
        df = df.drop(columns=cols_to_drop)
        print(f"  Removed {len(cols_to_drop)} columns. "
              f"Now has {df.shape[1]} columns.")
    return df

sold     = drop_duplicate_suffix_cols(sold,     'sold')
listings = drop_duplicate_suffix_cols(listings, 'listings')

# =========================================================================
# STEP 4 — Missing value handling
# =========================================================================

print("\n" + "=" * 65)
print("STEP 4 — MISSING VALUE HANDLING")
print("=" * 65)

for df, label in [(sold, 'sold'), (listings, 'listings')]:
    total       = len(df)
    missing_pct = (df.isnull().sum() / total * 100).sort_values(ascending=False)

    drop_cols = missing_pct[missing_pct > 90].index.tolist()
    print(f"\n  [{label}] Columns > 90% missing → DROP ({len(drop_cols)} cols):")
    for col in drop_cols:
        print(f"    • {col:<40} {missing_pct[col]:.1f}%")
    if drop_cols:
        df.drop(columns=drop_cols, inplace=True)
        print(f"  Dropped. Now has {df.shape[1]} columns.")

    mid_cols = missing_pct[(missing_pct > 50) & (missing_pct <= 90)]
    mid_cols = mid_cols[mid_cols.index.isin(df.columns)]
    print(f"\n  [{label}] Columns 50–90% missing → KEEP (action TBD):")
    if len(mid_cols) > 0:
        for col, pct in mid_cols.items():
            print(f"    • {col:<40} {pct:.1f}%")
    else:
        print("    None found.")

    if label == 'sold':
        sold = df
    else:
        listings = df

print(f"\n  [sold] Rows with missing ClosePrice:")
missing_close = sold[sold['ClosePrice'].isna()]
print(f"  Count: {len(missing_close)}")
if len(missing_close) > 0:
    show_cols = ['ListingKey', 'CloseDate', 'ListPrice',
                 'ClosePrice', 'CountyOrParish']
    show_cols = [c for c in show_cols if c in missing_close.columns]
    print(missing_close[show_cols].to_string(index=True))

# =========================================================================
# STEP 5 — Ensure numeric fields are properly typed
# Learned from teammate: scan ALL object columns automatically instead
# of a predefined list. Only convert if coercion creates NO new nulls
# (meaning every non-null value successfully parsed as a number).
# Exclude known ID/code columns to protect leading zeros.
# =========================================================================

print("\n" + "=" * 65)
print("STEP 5 — NUMERIC FIELD TYPE CHECK AND CONVERSION")
print("=" * 65)

def fix_numeric_types(df, label):
    converted       = []
    skipped_id      = []
    skipped_text    = []

    object_cols = [c for c in df.columns if df[c].dtype == object]

    for col in object_cols:
        if col in NUMERIC_CONVERSION_EXCLUDE:
            skipped_id.append(col)
            continue

        coerced          = pd.to_numeric(df[col], errors='coerce')
        original_missing = df[col].isna().sum()
        coerced_missing  = coerced.isna().sum()

        # Only convert if no NEW nulls were created — meaning every
        # non-null value successfully parsed as a number
        if coerced_missing == original_missing and coerced_missing < len(df):
            df[col] = coerced.astype('float64')
            converted.append(col)
        else:
            skipped_text.append(col)

    print(f"\n  [{label}]")
    print(f"    Converted to float64                : {converted if converted else 'none'}")
    print(f"    Skipped (ID/code, kept as text)     : {skipped_id}")
    print(f"    Skipped (contains text, left as-is) : {len(skipped_text)} columns")
    print(f"    Final dtype breakdown:")
    for dtype, count in df.dtypes.value_counts().items():
        print(f"      {str(dtype):<20} : {count}")
    return df

sold     = fix_numeric_types(sold,     'sold')
listings = fix_numeric_types(listings, 'listings')

# =========================================================================
# STEP 6 — Geographic flags
# Learned from teammate: add a SEPARATE missing coordinate flag.
# lat_0_flag / long_0_flag  → coordinate is exactly 0 (failed geocode)
# geo_missing_flag          → coordinate is null (no geocode at all)
# out_of_state_flag         → coordinate outside CA bounding box
#                             (missing coords are also flagged here since
#                              .between() returns False for NaN)
# =========================================================================

print("\n" + "=" * 65)
print("STEP 6 — GEOGRAPHIC FLAGS")
print("=" * 65)

for df, label in [(sold, 'sold'), (listings, 'listings')]:
    print(f"\n  [{label}]")

    if 'Latitude' not in df.columns or 'Longitude' not in df.columns:
        print("    Latitude/Longitude not found — skipping")
        continue

    # Missing coordinates (null — no geocode attempt succeeded)
    df['geo_missing_flag'] = df['Latitude'].isna() | df['Longitude'].isna()
    print(f"    geo_missing_flag  (lat or lon is null)  : "
          f"{df['geo_missing_flag'].sum():,}")

    # Zero coordinates (geocoder returned 0,0 as a sentinel null)
    df['lat_0_flag']  = df['Latitude'].notna()  & (df['Latitude']  == 0)
    df['long_0_flag'] = df['Longitude'].notna() & (df['Longitude'] == 0)
    print(f"    lat_0_flag        (Latitude == 0)        : "
          f"{df['lat_0_flag'].sum():,}")
    print(f"    long_0_flag       (Longitude == 0)       : "
          f"{df['long_0_flag'].sum():,}")

    # Outside California bounding box
    # .between() returns False for NaN, so missing coords are also
    # captured by out_of_state_flag via the negation
    in_ca_lat = df['Latitude'].between(*CA_LAT_RANGE)
    in_ca_lon = df['Longitude'].between(*CA_LON_RANGE)
    df['out_of_state_flag'] = ~(in_ca_lat & in_ca_lon)
    print(f"    out_of_state_flag (outside CA bbox)      : "
          f"{df['out_of_state_flag'].sum():,}")

    total     = len(df)
    any_issue = (df['geo_missing_flag'] | df['lat_0_flag'] |
                 df['long_0_flag']      | df['out_of_state_flag'])
    clean_geo = (~any_issue).sum()
    print(f"    Clean coordinates : {clean_geo:,} / {total:,} "
          f"({clean_geo/total*100:.1f}%)")

    if label == 'sold':
        sold = df
    else:
        listings = df

# =========================================================================
# STEP 7 — Remove invalid numeric values
# =========================================================================

print("\n" + "=" * 65)
print("STEP 7 — REMOVE INVALID NUMERIC VALUES")
print("=" * 65)

print("\n  [sold]")
if 'ClosePrice' in sold.columns:
    n    = (sold['ClosePrice'].notna() & (sold['ClosePrice'] <= 0)).sum()
    sold = sold[~(sold['ClosePrice'].notna() & (sold['ClosePrice'] <= 0))]
    print(f"    ClosePrice <= 0  removed : {n:,}")
if 'ListPrice' in sold.columns:
    n = (sold['ListPrice'].notna() & (sold['ListPrice'] <= 0)).sum()
    print(f"    ListPrice <= 0   count   : {n:,}  "
          f"(kept for now — pending discussion)")

print("\n  [listings]")
if 'ListPrice' in listings.columns:
    n        = (listings['ListPrice'].notna() &
                (listings['ListPrice'] <= 0)).sum()
    listings = listings[
        ~(listings['ListPrice'].notna() & (listings['ListPrice'] <= 0))
    ]
    print(f"    ListPrice <= 0   removed : {n:,}")
print(f"    ClosePrice               skipped (ignore in List dataset for now)")

BOTH_RULES = [
    ('LivingArea',            '<=', 0),
    ('BedroomsTotal',         '<',  0),
    ('BathroomsTotalInteger', '<',  0),
    ('DaysOnMarket',          '<',  0),
]

for df, label in [(sold, 'sold'), (listings, 'listings')]:
    print(f"\n  [{label}] — shared rules")
    for col, op, threshold in BOTH_RULES:
        if col not in df.columns:
            continue
        before_rule = len(df)
        mask = (df[col].notna() &
                (df[col] <= threshold if op == '<=' else df[col] < threshold))
        df   = df[~mask]
        removed = before_rule - len(df)
        print(f"    {col:<28} {op} {threshold}  removed: {removed:,}")
    if label == 'sold':
        sold = df
    else:
        listings = df

print(f"\n  Rows after Step 7:")
print(f"    sold     : {len(sold):,}")
print(f"    listings : {len(listings):,}")

# =========================================================================
# STEP 8 — Duplicate row check
# Logic learned from teammate:
# A listing can appear in more than one monthly export (e.g. an Active
# listing pulled in both March and April). This finds rows that match on
# every column except ListingKey/ListingKeyNumeric ("content duplicates").
#   - Same content + same ListingKey → truly the same record, keep one
#   - Same content + different ListingKey → legitimately different listings
#     that happen to look alike, keep both
# This is safer than drop_duplicates(subset=['ListingKey']) which would
# delete a record even if its data changed between months (e.g. price drop).
# =========================================================================

print("\n" + "=" * 65)
print("STEP 8 — DUPLICATE ROW CHECK")
print("=" * 65)

def dedupe_records(df, label):
    # Exclude identifier columns when checking for identical content
    # These are expected to differ across months and are not property data
    id_cols      = {'ListingKey', 'ListingKeyNumeric', 'year_month'}
    compare_cols = [c for c in df.columns if c not in id_cols]
    rows_before  = len(df)

    if 'ListingKey' not in df.columns:
        print(f"  [{label}] ListingKey not found — duplicate check skipped.")
        return df

    # Find every row that has at least one content-identical partner
    dup_mask = df.duplicated(subset=compare_cols, keep=False)
    dup_rows = df[dup_mask]

    to_drop_indices       = []
    n_true_dup_groups     = 0
    n_distinct_key_groups = 0

    if len(dup_rows) > 0:
        for _, group in dup_rows.groupby(compare_cols, dropna=False):
            if group['ListingKey'].nunique() == 1:
                # Same ListingKey → truly the same record, keep first only
                to_drop_indices.extend(group.index[1:])
                n_true_dup_groups += 1
            else:
                # Different ListingKey → different listings, keep all
                n_distinct_key_groups += 1

    cleaned = df.drop(index=to_drop_indices).reset_index(drop=True)

    print(f"\n  [{label}]")
    print(f"    Rows in content-duplicate groups           : {len(dup_rows):,}")
    print(f"    True duplicate groups (same key, keep one) : {n_true_dup_groups:,}")
    print(f"    Distinct-key groups (different key, keep all): {n_distinct_key_groups:,}")
    print(f"    Rows dropped                               : {len(to_drop_indices):,}")
    print(f"    Rows before: {rows_before:,}  →  after: {len(cleaned):,}")
    return cleaned

sold     = dedupe_records(sold,     'sold')
listings = dedupe_records(listings, 'listings')

# =========================================================================
# STEP 9 — Date consistency flags (flag only, not remove)
# Fixed from teammate's code: negative_timeline_flag now includes a
# third condition — PurchaseContractDate < ListingContractDate
# (contract signed before the property was even listed), in addition to
# the original two conditions.
# =========================================================================

print("\n" + "=" * 65)
print("STEP 9 — DATE CONSISTENCY FLAGS (flag only, not remove)")
print("=" * 65)

for df, label in [(sold, 'sold'), (listings, 'listings')]:
    print(f"\n  [{label}]")

    # Condition 1: close date is before listing date
    if 'ListingContractDate' in df.columns and 'CloseDate' in df.columns:
        mask = df['ListingContractDate'].notna() & df['CloseDate'].notna()
        df['listing_after_close_flag'] = (
            mask & (df['ListingContractDate'] > df['CloseDate'])
        )
        print(f"    listing_after_close_flag   "
              f"(CloseDate < ListingContractDate)  : "
              f"{df['listing_after_close_flag'].sum():,}")
    else:
        df['listing_after_close_flag'] = False

    # Condition 2: close date is before purchase contract date
    if 'PurchaseContractDate' in df.columns and 'CloseDate' in df.columns:
        mask = df['PurchaseContractDate'].notna() & df['CloseDate'].notna()
        df['purchase_after_close_flag'] = (
            mask & (df['PurchaseContractDate'] > df['CloseDate'])
        )
        print(f"    purchase_after_close_flag  "
              f"(CloseDate < PurchaseContractDate) : "
              f"{df['purchase_after_close_flag'].sum():,}")
    else:
        df['purchase_after_close_flag'] = False

    # Condition 3: purchase contract date is before listing date
    # (offer accepted before the property was even listed)
    if 'PurchaseContractDate' in df.columns and 'ListingContractDate' in df.columns:
        mask = df['PurchaseContractDate'].notna() & df['ListingContractDate'].notna()
        purchase_before_listing = (
            mask & (df['PurchaseContractDate'] < df['ListingContractDate'])
        )
        print(f"    purchase_before_listing    "
              f"(PurchaseContractDate < ListingContractDate) : "
              f"{purchase_before_listing.sum():,}")
    else:
        purchase_before_listing = pd.Series(False, index=df.index)

    # negative_timeline_flag = ANY of the three conditions above
    df['negative_timeline_flag'] = (
        df['listing_after_close_flag']  |
        df['purchase_after_close_flag'] |
        purchase_before_listing
    )
    print(f"    negative_timeline_flag     "
          f"(any of the 3 above)               : "
          f"{df['negative_timeline_flag'].sum():,}")

    if label == 'sold':
        sold = df
    else:
        listings = df

# =========================================================================
# Save outputs
# =========================================================================

print("\n" + "=" * 65)
print("SAVING CLEANED DATASETS")
print("=" * 65)

sold.to_csv(os.path.join(PATH, 'all_sold_cleaned.csv'),
            index=False, encoding='utf-8')
listings.to_csv(os.path.join(PATH, 'all_listings_cleaned.csv'),
                index=False, encoding='utf-8')

print(f"  Saved: all_sold_cleaned.csv     — "
      f"{len(sold):,} rows  |  {sold.shape[1]} columns")
print(f"  Saved: all_listings_cleaned.csv — "
      f"{len(listings):,} rows  |  {listings.shape[1]} columns")

# =========================================================================
# Final summary
# =========================================================================

print("\n" + "=" * 65)
print("FINAL SUMMARY")
print("=" * 65)
print(f"\nRow counts:")
print(f"  sold     : {len(sold_raw):,} raw → {len(sold):,} cleaned")
print(f"  listings : {len(listings_raw):,} raw → {len(listings):,} cleaned")

print(f"\nColumn counts:")
print(f"  sold     : {sold_raw.shape[1]} raw → {sold.shape[1]} cleaned")
print(f"  listings : {listings_raw.shape[1]} raw → {listings.shape[1]} cleaned")

print(f"\nSold final dtype breakdown:")
print(sold.dtypes.value_counts().to_string())

print(f"\nListing final dtype breakdown:")
print(listings.dtypes.value_counts().to_string())

print(f"\nGeographic data quality summary:")
for label, df in [("Sold", sold), ("Listing", listings)]:
    if 'out_of_state_flag' in df.columns:
        print(f"  [{label}] geo_missing: {df['geo_missing_flag'].sum():,}  "
              f"lat_0: {df['lat_0_flag'].sum():,}  "
              f"long_0: {df['long_0_flag'].sum():,}  "
              f"out_of_state: {df['out_of_state_flag'].sum():,}  "
              f"(of {len(df):,} total rows)")

print(f"\nDate consistency flag counts:")
for label, df in [("Sold", sold), ("Listing", listings)]:
    if 'negative_timeline_flag' in df.columns:
        print(f"  [{label}] listing_after_close: {df['listing_after_close_flag'].sum():,}  "
              f"purchase_after_close: {df['purchase_after_close_flag'].sum():,}  "
              f"negative_timeline: {df['negative_timeline_flag'].sum():,}")

Loading datasets...
  all_sold     : 448,033 rows  |  84 columns
  all_listings : 615,739 rows  |  84 columns

STEP 1 — DATE FORMAT CONVERSION

  [sold]
    CloseDate                           → datetime64[ns]  missing before=0, after=0 (0 failed to parse → NaT)
    PurchaseContractDate                → datetime64[ns]  missing before=198, after=198 (0 failed to parse → NaT)
    ListingContractDate                 → datetime64[ns]  missing before=1, after=1 (0 failed to parse → NaT)
    ContractStatusChangeDate            → datetime64[ns]  missing before=589, after=589 (0 failed to parse → NaT)

  [listings]
    CloseDate                           → datetime64[ns]  missing before=445,084, after=445,084 (0 failed to parse → NaT)
    PurchaseContractDate                → datetime64[ns]  missing before=326,674, after=326,674 (0 failed to parse → NaT)
    ListingContractDate                 → datetime64[ns]  missing before=0, after=0 (0 failed to parse → NaT)
    ContractStatusChangeDate   